# Classifier eval: Luna vs Sol

Answers the question from `docs/DECISIONS.md` D-29: does `gpt-6-sol` (~20x `gpt-6-luna`'s price)
earn its cost on the classifier call, the highest-volume LLM call in the pipeline (one per
`(article, topic)` candidate)? Jev is cut for this phase (`docs/phases/04-quality.md`'s own
"cut first" line, `docs/phases/README.md`'s "Jev optional") -- no Typesafe/Jev SDK or request
shape exists yet (see D-09), so this notebook compares Luna vs Sol only.

**Before running this notebook:** `eval/articles.csv` must exist and be fully labeled --
`label_relevant` and `label_newsworthy` filled in with `y`/`n` for every row, saved as
"CSV UTF-8" from Excel. See `docs/phases/04-quality.md` Part B steps 1-3.

Run from the repo root: `uv run --project backend jupyter lab`, then open this file (D-11: `app` is
importable with no path hacks, since `backend` is `uv_build`-installed editable into
`backend/.venv`). Classifier calls are cached to `results/scores_<name>.jsonl` and reused on
re-run, so "Restart & Run All" doesn't spend money twice and scores stay stable across runs
(D-29 point 3). Trust this notebook's numbers only after a clean "Restart & Run All".

In [ ]:
import json
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from app.adapters.classifier.openai import LLMClassifier
from app.adapters.llm.openai import OpenAILLM
from app.config import get_settings
from app.db import session_scope
from app.models import Article, Preferences, User
from app.pipeline.rank import Candidate, select_stories
from app.schemas import InterestProfile, Topic
from sqlalchemy import select

settings = get_settings()
ARTICLES_CSV = Path("articles.csv")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
MODELS = {"luna": "gpt-6-luna", "sol": "gpt-6-sol"}
THRESHOLD = 0.5

## Metric functions, checked against a hand-computed toy example before touching real scores

In [ ]:
def accuracy(y_true, y_pred_binary):
    return sum(int(t == p) for t, p in zip(y_true, y_pred_binary)) / len(y_true)


def precision(y_true, y_pred_binary):
    tp = sum(1 for t, p in zip(y_true, y_pred_binary) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(y_true, y_pred_binary) if t == 0 and p == 1)
    return tp / (tp + fp) if (tp + fp) else float("nan")


def recall(y_true, y_pred_binary):
    tp = sum(1 for t, p in zip(y_true, y_pred_binary) if t == 1 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred_binary) if t == 1 and p == 0)
    return tp / (tp + fn) if (tp + fn) else float("nan")


def roc_auc(y_true, y_score):
    """Rank-based AUC (probability a random positive scores above a random
    negative, ties count half) -- no sklearn needed for ~60 rows."""
    pos = [s for t, s in zip(y_true, y_score) if t == 1]
    neg = [s for t, s in zip(y_true, y_score) if t == 0]
    if not pos or not neg:
        return float("nan")
    wins = sum((p > n) + 0.5 * (p == n) for p in pos for n in neg)
    return wins / (len(pos) * len(neg))


def binarize(scores, threshold=THRESHOLD):
    return [1 if s >= threshold else 0 for s in scores]

In [ ]:
# Toy example, hand-computed, checked before trusting the functions on real scores.
_toy_true = [1, 1, 0, 0]
_toy_pred = [1, 0, 0, 0]  # 1 TP, 1 FN, 2 TN -> acc=3/4, precision=1/1=1.0, recall=1/2=0.5
assert accuracy(_toy_true, _toy_pred) == 0.75
assert precision(_toy_true, _toy_pred) == 1.0
assert recall(_toy_true, _toy_pred) == 0.5

_toy_scores = [0.9, 0.4, 0.3, 0.8]  # pos scores .9/.4, neg scores .3/.8
# pairs: (.9,.3) win, (.9,.8) win, (.4,.3) win, (.4,.8) loss -> 3/4 = 0.75
assert abs(roc_auc(_toy_true, _toy_scores) - 0.75) < 1e-9
print("metric functions match hand-computed toy values")

## Load the labeled CSV

Joins are always on `(article_id, fetched_topic)`, kept as explicit DataFrame columns and used as
dict keys throughout -- never on row position (D-29 point 2), so a partial sort in Excel can't
silently misalign ground truth.

In [ ]:
df = pd.read_csv(ARTICLES_CSV, encoding="utf-8-sig")
unlabeled = df["label_relevant"].isna() | df["label_newsworthy"].isna()
if unlabeled.any():
    raise SystemExit(
        f"{unlabeled.sum()} of {len(df)} rows in {ARTICLES_CSV} are still unlabeled -- "
        "label_relevant and label_newsworthy must be 'y' or 'n' for every row. Open it in Excel, "
        'fill in the label columns only, save as "CSV UTF-8", then re-run this notebook.'
    )
df["label_relevant_bin"] = df["label_relevant"].str.strip().str.lower().map({"y": 1, "n": 0})
df["label_newsworthy_bin"] = df["label_newsworthy"].str.strip().str.lower().map({"y": 1, "n": 0})
assert df["label_relevant_bin"].notna().all(), "label_relevant must be exactly 'y' or 'n'"
assert df["label_newsworthy_bin"].notna().all(), "label_newsworthy must be exactly 'y' or 'n'"
print(f"{len(df)} labeled rows across {df['fetched_topic'].nunique()} topics:")
df.groupby("fetched_topic").size()

## Transient Article objects and the eval user's real profile

No DB writes -- `Article` rows are built straight from the CSV and never added to a session. The
profile is read from the eval user's saved preferences (a DB read, not a write) so topic
descriptions/include/exclude match what `rank.py` actually scores against in production.

In [ ]:
def make_article(row: pd.Series) -> Article:
    published_at = None
    if pd.notna(row["published_at"]) and row["published_at"]:
        published_at = datetime.fromisoformat(row["published_at"])
    highlights = []
    if pd.notna(row["highlights"]) and row["highlights"]:
        highlights = str(row["highlights"]).split(" | ")
    return Article(
        id=int(row["article_id"]),
        url_hash=f"eval-{row['article_id']}",
        url="",
        outlet=row["outlet"] if pd.notna(row["outlet"]) else None,
        title=row["title"] if pd.notna(row["title"]) else None,
        published_at=published_at,
        highlights=highlights,
    )


with session_scope() as db:
    eval_user = db.scalars(select(User).where(User.email == settings.seed_eval_user_email)).one()
    prefs = db.get(Preferences, eval_user.id)
    profile = InterestProfile.model_validate(prefs.interest_profile)

_topics_by_name = {t.name: t for t in profile.topics}


def profile_for_topic(topic_name: str) -> InterestProfile:
    t = _topics_by_name.get(topic_name) or Topic(
        name=topic_name, description=f"News about {topic_name}"
    )
    return InterestProfile(topics=[t], avoid=profile.avoid)


print(f"eval user profile: {[t.name for t in profile.topics]}")

## Score with the production classifier class

`LLMClassifier` wraps the real `OpenAILLM` adapter -- exactly what `rank.py` calls in production,
just pointed at each model in turn via a settings copy (`LLMClassifier` reads
`settings.model_classifier`, so `model_copy(update=...)` swaps the model without touching env
vars). `recent_headlines=[]` per the phase doc (this eval set has no episode history behind it).

In [ ]:
llm = OpenAILLM(settings)


def make_classifier(model: str) -> LLMClassifier:
    return LLMClassifier(llm, settings.model_copy(update={"model_classifier": model}))


def cache_path(name: str) -> Path:
    return RESULTS_DIR / f"scores_{name}.jsonl"


def load_cache(name: str) -> dict[tuple[int, str], dict]:
    path = cache_path(name)
    if not path.exists():
        return {}
    cached = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        cached[(record["article_id"], record["fetched_topic"])] = record
    return cached


def score_classifier(name: str, model: str) -> dict[tuple[int, str], dict]:
    cached = load_cache(name)
    classifier = make_classifier(model)
    new_count = 0
    with cache_path(name).open("a", encoding="utf-8") as f:
        for _, row in df.iterrows():
            key = (int(row["article_id"]), row["fetched_topic"])
            if key in cached:
                continue
            article = make_article(row)
            result, usage = classifier.score(
                article,
                profile_for_topic(row["fetched_topic"]),
                row["fetched_topic"],
                recent_headlines=[],
            )
            record = {
                "article_id": key[0],
                "fetched_topic": key[1],
                "relevance": result.relevance,
                "newsworthy": result.newsworthy,
                "cost_usd": usage.cost_usd,
                "latency_ms": usage.latency_ms,
            }
            f.write(json.dumps(record) + "\n")
            cached[key] = record
            new_count += 1
    total_cost = sum(v["cost_usd"] for v in cached.values())
    print(f"{name}: {new_count} new, {len(cached)} total cached, ${total_cost:.4f} spent so far")
    return cached

In [ ]:
luna_scores = score_classifier("luna", MODELS["luna"])
sol_scores = score_classifier("sol", MODELS["sol"])
_spent = sum(v["cost_usd"] for v in luna_scores.values()) + sum(
    v["cost_usd"] for v in sol_scores.values()
)
print(f"total spend this run: ${_spent:.4f}")

In [ ]:
def with_scores(base: pd.DataFrame, scores: dict) -> pd.DataFrame:
    out = base.copy()
    out["relevance"] = [
        scores[(int(r["article_id"]), r["fetched_topic"])]["relevance"] for _, r in base.iterrows()
    ]
    out["newsworthy"] = [
        scores[(int(r["article_id"]), r["fetched_topic"])]["newsworthy"] for _, r in base.iterrows()
    ]
    return out


luna_df = with_scores(df, luna_scores)
sol_df = with_scores(df, sol_scores)

## Metrics: relevance (accuracy/precision/recall/ROC-AUC), newsworthy (precision/recall/ROC-AUC), latency, cost

In [ ]:
def summarize(name: str, scored_df: pd.DataFrame, cache: dict) -> dict:
    y_relevant = scored_df["label_relevant_bin"].tolist()
    y_newsworthy = scored_df["label_newsworthy_bin"].tolist()
    relevance_scores = scored_df["relevance"].tolist()
    newsworthy_scores = scored_df["newsworthy"].tolist()
    latencies = sorted(v["latency_ms"] for v in cache.values())
    total_cost = sum(v["cost_usd"] for v in cache.values())

    return {
        "classifier": name,
        "n": len(scored_df),
        "relevance_accuracy": accuracy(y_relevant, binarize(relevance_scores)),
        "relevance_precision": precision(y_relevant, binarize(relevance_scores)),
        "relevance_recall": recall(y_relevant, binarize(relevance_scores)),
        "relevance_roc_auc": roc_auc(y_relevant, relevance_scores),
        "newsworthy_precision": precision(y_newsworthy, binarize(newsworthy_scores)),
        "newsworthy_recall": recall(y_newsworthy, binarize(newsworthy_scores)),
        "newsworthy_roc_auc": roc_auc(y_newsworthy, newsworthy_scores),
        "latency_p50_ms": latencies[len(latencies) // 2],
        "latency_p95_ms": latencies[min(len(latencies) - 1, int(len(latencies) * 0.95))],
        "cost_per_100": total_cost / len(cache) * 100,
    }


luna_summary = summarize("luna", luna_df, luna_scores)
sol_summary = summarize("sol", sol_df, sol_scores)
summary_df = pd.DataFrame([luna_summary, sol_summary]).set_index("classifier")
summary_df

## Selection agreement

Feeds each classifier's own predicted scores through the production `select_stories` (not a
reimplemented top-N) and compares the resulting selected-story sets.
`target_minutes=10` -> `story_count_for(10) == 8` stories, matching the phase doc's "8 stories".

In [ ]:
_NOW = datetime.now(UTC)


def candidates_from(scored_df: pd.DataFrame) -> list[Candidate]:
    cands = []
    for _, r in scored_df.iterrows():
        published_at = None
        if pd.notna(r["published_at"]) and r["published_at"]:
            published_at = datetime.fromisoformat(r["published_at"])
        cands.append(
            Candidate(
                article_id=int(r["article_id"]),
                topic=r["fetched_topic"],
                relevance=r["relevance"],
                newsworthy=r["newsworthy"],
                already_covered=False,
                published_at=published_at,
            )
        )
    return cands


luna_picks = select_stories(candidates_from(luna_df), target_minutes=10, now=_NOW)
sol_picks = select_stories(candidates_from(sol_df), target_minutes=10, now=_NOW)
luna_selected = {s.article_id for s in luna_picks}
sol_selected = {s.article_id for s in sol_picks}
union = luna_selected | sol_selected
agreement = len(luna_selected & sol_selected) / len(union) if union else float("nan")
print(f"selection agreement (Jaccard over selected article ids): {agreement:.2f}")
print(f"luna-only: {luna_selected - sol_selected}")
print(f"sol-only: {sol_selected - luna_selected}")

## Default classifier decision

Rule fixed before looking at results (`docs/DECISIONS.md` D-29 point 4): Luna stays the default
unless selection agreement with Sol is low **and** Sol's gain on the axis where they differ most is
material. Thresholds, also fixed here rather than tuned to the outcome:

In [ ]:
LOW_AGREEMENT_THRESHOLD = 0.7  # selected-story Jaccard overlap below this counts as "low"
MATERIAL_GAIN_THRESHOLD = 0.10  # a >=10-point gain on accuracy/ROC-AUC counts as "material"
_COMPARISON_AXES = ["relevance_accuracy", "relevance_roc_auc", "newsworthy_roc_auc"]


def biggest_gain(a: dict, b: dict) -> tuple[str, float]:
    gains = {axis: b[axis] - a[axis] for axis in _COMPARISON_AXES}
    best_axis = max(gains, key=gains.get)
    return best_axis, gains[best_axis]


low_agreement = agreement < LOW_AGREEMENT_THRESHOLD
best_axis, gain = biggest_gain(luna_summary, sol_summary)
material_gain = gain >= MATERIAL_GAIN_THRESHOLD

if low_agreement and material_gain:
    default_classifier = "sol"
    reason = (
        f"selection agreement is {agreement:.2f} (< {LOW_AGREEMENT_THRESHOLD}) and Sol gains "
        f"{gain:+.2f} on {best_axis} (>= {MATERIAL_GAIN_THRESHOLD}) -- worth ~20x the cost here."
    )
else:
    default_classifier = "luna"
    reason = (
        f"selection agreement is {agreement:.2f}, Sol's biggest gain is {gain:+.2f} on "
        f"{best_axis} -- not both low-agreement and material, so Luna's ~20x lower cost "
        "wins by default."
    )

print(f"default classifier: {default_classifier}")
print(f"reason: {reason}")

## Plots for solution.md: cost vs quality, where the classifiers disagree

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for s in (luna_summary, sol_summary):
    axes[0].scatter(s["cost_per_100"], s["relevance_roc_auc"], label=s["classifier"])
    axes[0].annotate(s["classifier"], (s["cost_per_100"], s["relevance_roc_auc"]))
axes[0].set_xlabel("cost per 100 articles ($)")
axes[0].set_ylabel("relevance ROC-AUC")
axes[0].set_title("cost vs quality")

disagreement = (luna_df["relevance"] - sol_df["relevance"]).abs()
axes[1].hist(disagreement, bins=10)
axes[1].set_xlabel("|luna relevance - sol relevance|")
axes[1].set_title("where the classifiers disagree")

fig.tight_layout()
fig.savefig(RESULTS_DIR / "cost_vs_quality.png", dpi=120)
plt.show()

## Write results for the dashboard (phase 07) and `solution.md`

In [ ]:
today = datetime.now(UTC).date().isoformat()
results_payload = {
    "date": today,
    "classifiers": {"luna": luna_summary, "sol": sol_summary},
    "selection_agreement": agreement,
    "default_classifier": default_classifier,
    "default_reason": reason,
}
(RESULTS_DIR / f"{today}.json").write_text(json.dumps(results_payload, indent=2), encoding="utf-8")

_MD_COLUMNS = [
    "classifier",
    "n",
    "rel. acc",
    "rel. P",
    "rel. R",
    "rel. ROC-AUC",
    "news P",
    "news R",
    "news ROC-AUC",
    "p50 ms",
    "p95 ms",
    "$/100",
]


def md_row(s: dict) -> str:
    values = [
        s["classifier"],
        s["n"],
        f"{s['relevance_accuracy']:.2f}",
        f"{s['relevance_precision']:.2f}",
        f"{s['relevance_recall']:.2f}",
        f"{s['relevance_roc_auc']:.2f}",
        f"{s['newsworthy_precision']:.2f}",
        f"{s['newsworthy_recall']:.2f}",
        f"{s['newsworthy_roc_auc']:.2f}",
        s["latency_p50_ms"],
        s["latency_p95_ms"],
        f"${s['cost_per_100']:.3f}",
    ]
    return "| " + " | ".join(str(v) for v in values) + " |"


md_lines = [
    "# Classifier eval results",
    "",
    f"Default classifier: **{default_classifier}** -- {reason}",
    "",
    "| " + " | ".join(_MD_COLUMNS) + " |",
    "|" + "---|" * len(_MD_COLUMNS),
    md_row(luna_summary),
    md_row(sol_summary),
    "",
    f"Selection agreement (Jaccard, target_minutes=10 -> 8 stories): {agreement:.2f}",
]
(RESULTS_DIR / "latest.md").write_text("\n".join(md_lines), encoding="utf-8")
print(f"wrote {RESULTS_DIR / (today + '.json')} and {RESULTS_DIR / 'latest.md'}")